<a href="https://colab.research.google.com/github/sdmadhav/mtp_NUMERICAL_CLAIM_VERIFICATION/blob/main/3_encoder_logic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!gdown --id 1jU264txlUGx4n0lWhb4VP41qBbPk-u_N

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1jU264txlUGx4n0lWhb4VP41qBbPk-u_N
From (redirected): https://drive.google.com/uc?id=1jU264txlUGx4n0lWhb4VP41qBbPk-u_N&confirm=t&uuid=c4591c7c-f84a-4684-acdd-ce650e64d7a7
To: /content/Processed_complete_dataset.json
100% 252M/252M [00:01<00:00, 128MB/s]


In [ ]:
import pandas as pd
df = pd.read_json("Processed_complete_dataset.json")
df.head()

,claim,quantemp_evidences,label,Category,our_evidences,taxonomy,reranked_our_evidences
0,"""The non-partisan Congressional Budget Office ...",[{'questions': 'did the congressional budget o...,Conflicting,test,[{'question': 'Who concluded ObamaCare will co...,Statistical,[{'questions': 'Who concluded ObamaCare will c...
1,"""More than 50 percent of immigrants from (El S...",[{'questions': 'do more than 50 percent of imm...,True,test,[{'question': 'Which country has over 50 perce...,Statistical,[{'questions': 'Which country has over 50 perc...
2,UK government banned Covid vaccine for childre...,[{'questions': 'has the uk government banned t...,False,test,[{'question': 'Which government banned Covid v...,Temporal,[{'questions': 'Which government banned Covid ...
3,"""[In 2014-2015] coverage for the rotavirus vac...",[{'questions': 'did the coverage for the rotav...,False,test,[{'question': 'Which vaccine had 91.5% coverag...,Statistical,[{'questions': 'Which vaccine had 91.5% covera...
4,"In September 2021, the U.K. government announc...",[{'questions': 'is the u.k. government plannin...,True,test,[{'question': 'Which government announced its ...,Temporal,[{'questions': 'Which government announced its...


In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define source file path and destination folder path
source_file = '/content/final_reranked_temporal+page_content_refined.json'
destination_folder = '/content/drive/MyDrive/claim_verification_final'

# Create the destination folder if it doesn't exist
os.makedirs(destination_folder, exist_ok=True)

# Copy the file
!cp "{source_file}" "{destination_folder}/"

print(f"File '{source_file}' copied to '{destination_folder}'")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
File '/content/final_reranked_temporal+page_content_refined.json' copied to '/content/drive/MyDrive/claim_verification_final'


In [ ]:
import pandas as pd
df = pd.read_json("/content/drive/MyDrive/claim_verification_final/final_reranked_temporal+page_content_refined.json")

In [ ]:
df

,claim,label,category,taxonomy,reranked_our_evidences
0,A russian serviceman personally neutralized n...,False,train,Statistical,[{'questions': 'Which russian soldier neutrali...
1,Alex Soros stated that he is 100 times worse ...,False,validation,Comparison,[{'questions': 'Who is 100 times worse than hi...
2,"In Israel, with a population of 8 million cit...",False,train,Statistical,[{'questions': 'Which country had 3 million ci...
3,Kyiv cut off gas in the territory of Luhansk ...,False,train,Temporal,[{'questions': 'Which country cut off gas in t...
4,Pfizer’s CEO says one of the company’s goals ...,False,train,Statistical,[{'questions': 'Which company has a goal of re...
...,...,...,...,...,...
15410,„HalleLeaks“ erfindet Flüchtlings-Täter im Mor...,False,train,Interval,[{'questions': 'Which article had the police B...
15411,„Ärzte für Aufklärung“ verbreiten in ihrer Vid...,Half True/False,validation,Interval,[{'questions': 'Which group has released a vid...
15412,… we have a chance to reach the 5% positivity ...,False,train,Interval,[{'questions': 'Which country has 5% positivit...
15413,₹1.85 crore balance had to be paid by SPB’s fa...,False,train,Statistical,[{'questions': 'Which body had 1.85 crore bala...


In [ ]:
df_train = df[df.category != 'test']
df_test = df[df.category == 'test']

In [ ]:
"""
=============================================================================
FACT VERIFICATION PIPELINE WITH INTERPRETABLE QUESTION CONTRIBUTION
=============================================================================
Architecture:
  Stage 1 : 3 separate BERT encoders  → one per (claim, question_i, evidence_i)
  Stage 2 : Shared projection FNN     → maps each encoder [CLS] to hidden dim
  Stage 3 : Attention over 3 reps    → learn question importance weights
  Stage 4 : Weighted sum + classifier → 3-class veracity prediction

Output  : logits (4,) + attention_weights (3,) per sample
=============================================================================
"""

# ─────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import ast
import re
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# 1.  CONFIGURATION  (change these to tune the run)
# ─────────────────────────────────────────────────────────────────────────────
CFG = dict(
    model_name     = "bert-base-uncased",   # any HF encoder works
    max_length     = 256,                   # token budget per input pair
    hidden_dim     = 256,                   # projection + attention dim
    dropout        = 0.1,
    num_labels     = 3,
    batch_size     = 8,
    num_epochs     = 3,
    lr             = 2e-5,
    warmup_ratio   = 0.1,
    seed           = 42,
    device         = "cuda" if torch.cuda.is_available() else "cpu",
)

LABEL2ID = {
    "False":         0,
    "True":          1,
    "Conflicting":   2,
    "Half True/False": 2,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])
print(f"[CONFIG] Using device: {CFG['device']}")


# ─────────────────────────────────────────────────────────────────────────────
# 2.  PREPROCESSING HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def safe_parse_column(value):
    """
    Safely parse a cell that may be:
      - already a dict / list
      - a JSON/Python string representation of one
      - None / NaN
    Returns a dict or None.
    """
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        try:
            return ast.literal_eval(value)
        except Exception:
            return None
    return None


def extract_qa_evidence(parsed, q_idx: int, top_k: int = 1):
    """
    From a parsed dict for one question slot, return:
      (question_text, evidence_text)

    parsed expected shape:
      {"questions": "...", "top_k_doc": ["doc1", "doc2", ...]}

    q_idx  : index into the 3-element list of question dicts (for logging)
    top_k  : how many evidence docs to concatenate (currently 1)
    """
    if parsed is None:
        return "", ""

    question = parsed.get("questions", "") or ""
    docs     = parsed.get("top_k_doc", []) or []

    # Take only top-1 evidence (as per Stage 1 requirement)
    evidence = " ".join(docs[:top_k]) if docs else ""
    return question.strip(), evidence.strip()


def clean_label(raw_label) -> str:
    """Strip whitespace/newlines and normalise label string."""
    if not isinstance(raw_label, str):
        raw_label = str(raw_label)
    label = raw_label.strip()
    # Catch common variants
    if re.search(r"half.?true", label, re.I) or re.search(r"half.?false", label, re.I):
        return "Half True/False"
    for canonical in LABEL2ID:
        if label.lower() == canonical.lower():
            return canonical
    return label  # will be filtered out later


def preprocess_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert raw df into a clean df with columns:
      claim, q1, ev1, q2, ev2, q3, ev3, label_id
    Rows with unknown labels are dropped.
    """
    records = []
    skipped = 0

    for idx, row in df.iterrows():
        # --- label ---
        raw_label = row.get("label", "")
        label_str = clean_label(raw_label)
        if label_str not in LABEL2ID:
            skipped += 1
            continue
        label_id = LABEL2ID[label_str]

        # --- claim ---
        claim = str(row.get("claim", "")).strip()

        # --- question columns (handle list or 3 separate cols) ---
        # Support two layouts:
        #   Layout A: df["column"] = list of 3 dicts
        #   Layout B: df["q1"], df["q2"], df["q3"] already separate
        q_slots = []
        if "reranked_our_evidences" in row and isinstance(row["reranked_our_evidences"], (list, str)):
            raw_col = row["reranked_our_evidences"]
            if isinstance(raw_col, str):
                try:
                    raw_col = ast.literal_eval(raw_col)
                except Exception:
                    raw_col = []
            if not isinstance(raw_col, list):
                raw_col = []
            for i in range(3):
                q_slots.append(safe_parse_column(raw_col[i]) if i < len(raw_col) else None)
        else:
            for col in ["q1", "q2", "q3"]:
                q_slots.append(safe_parse_column(row.get(col)))

        # extract (question, evidence) for each slot
        pairs = [extract_qa_evidence(slot, i) for i, slot in enumerate(q_slots)]
        # pad to exactly 3
        while len(pairs) < 3:
            pairs.append(("", ""))

        record = {
            "claim":    claim,
            "q1":       pairs[0][0], "ev1": pairs[0][1],
            "q2":       pairs[1][0], "ev2": pairs[1][1],
            "q3":       pairs[2][0], "ev3": pairs[2][1],
            "label_id": label_id,
        }
        records.append(record)

    print(f"[PREPROCESS] Kept {len(records)} rows, skipped {skipped} (unknown labels).")
    return pd.DataFrame(records)


# ─────────────────────────────────────────────────────────────────────────────
# 3.  TOKENIZATION HELPER
# ─────────────────────────────────────────────────────────────────────────────

def tokenize_pair(tokenizer, claim: str, question: str, evidence: str,
                  max_length: int) -> dict:
    """
    Format:  text_a = claim
             text_b = question + [SEP] + evidence
    Returns a dict with input_ids, attention_mask (and token_type_ids if the
    tokenizer produces them).
    """
    text_a = claim
    # Concatenate question and evidence with a separator for text_b
    text_b = (question + " [SEP] " + evidence).strip() if (question or evidence) else " "

    encoded = tokenizer(
        text_a,
        text_b,
        max_length     = max_length,
        padding        = "max_length",
        truncation     = True,
        return_tensors = "pt",
    )
    # Squeeze batch dim → shape (seq_len,) tensors
    return {k: v.squeeze(0) for k, v in encoded.items()}


# ─────────────────────────────────────────────────────────────────────────────
# 4.  DATASET CLASS
# ─────────────────────────────────────────────────────────────────────────────

class FactVerificationDataset(Dataset):
    """
    Returns q1_inputs, q2_inputs, q3_inputs, label for each claim.
    Each q*_inputs is a dict: {input_ids, attention_mask, [token_type_ids]}.
    """

    def __init__(self, df_clean: pd.DataFrame, tokenizer, max_length: int):
        self.df         = df_clean.reset_index(drop=True)
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        claim = row["claim"]

        q1_inputs = tokenize_pair(self.tokenizer, claim, row["q1"], row["ev1"], self.max_length)
        q2_inputs = tokenize_pair(self.tokenizer, claim, row["q2"], row["ev2"], self.max_length)
        q3_inputs = tokenize_pair(self.tokenizer, claim, row["q3"], row["ev3"], self.max_length)

        # Label must be a plain int tensor
        label = torch.tensor(int(row["label_id"]), dtype=torch.long)

        return q1_inputs, q2_inputs, q3_inputs, label


def collate_fn(batch):
    """Stack individual sample dicts into batch dicts."""
    q1_list, q2_list, q3_list, labels = zip(*batch)

    def stack_dicts(dict_list):
        return {k: torch.stack([d[k] for d in dict_list]) for k in dict_list[0]}

    return (
        stack_dicts(q1_list),
        stack_dicts(q2_list),
        stack_dicts(q3_list),
        torch.stack(labels),
    )


# ─────────────────────────────────────────────────────────────────────────────
# 5.  MODEL
# ─────────────────────────────────────────────────────────────────────────────

class QuestionEncoder(nn.Module):
    """
    Wraps a BERT-like encoder.
    Returns the [CLS] token representation: shape (batch, encoder_hidden).
    Handles models that do / do not use token_type_ids via **kwargs.
    """

    def __init__(self, model_name: str):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.hidden_size = self.encoder.config.hidden_size

    def forward(self, **inputs):
        # Safely drop token_type_ids if the model doesn't support them
        # (e.g. RoBERTa). BERT does support them, but this makes the code
        # robust to any HF encoder.
        supported_keys = self.encoder.forward.__code__.co_varnames
        if "token_type_ids" not in supported_keys:
            inputs.pop("token_type_ids", None)

        outputs = self.encoder(**inputs)
        # outputs.last_hidden_state shape: (batch, seq_len, hidden)
        cls_repr = outputs.last_hidden_state[:, 0, :]   # (batch, hidden)
        return cls_repr


class AttentionFusion(nn.Module):
    """
    Stage 3: Given 3 projected representations (batch, 3, hidden_dim),
    compute a scalar attention score per question, softmax → weights,
    then return the weighted sum.

    The attention score is a single linear layer (additive style).
    """

    def __init__(self, hidden_dim: int):
        super().__init__()
        # Learnable query vector for scoring each question repr
        self.attention_vector = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, stacked: torch.Tensor):
        """
        stacked : (batch, 3, hidden_dim)
        returns : weighted_sum (batch, hidden_dim),
                  weights      (batch, 3)   ← interpretability hook
        """
        # scores: (batch, 3, 1) → squeeze → (batch, 3)
        scores  = self.attention_vector(stacked).squeeze(-1)
        weights = F.softmax(scores, dim=-1)                  # (batch, 3)

        # Weighted sum: (batch, 3, 1) * (batch, 3, hidden_dim) → sum over 3
        weighted_sum = (weights.unsqueeze(-1) * stacked).sum(dim=1)  # (batch, hidden_dim)
        return weighted_sum, weights


class FactVerificationModel(nn.Module):
    """
    Full 4-stage fact verification model.

    Stage 1 : 3 shared-weight encoders (same BERT, called 3 times)
              Using shared weights reduces parameter count and regularises
              training. Swap to 3 independent encoders if preferred.
    Stage 2 : Shared projection FNN  encoder_hidden → hidden_dim
    Stage 3 : Attention over the 3 projected representations
    Stage 4 : Weighted sum → dropout → classifier → 4 logits
    """

    def __init__(self, model_name: str, hidden_dim: int,
                 num_labels: int, dropout: float):
        super().__init__()

        # ── Stage 1: encoder (shared weights across 3 question slots) ──────
        self.encoder = QuestionEncoder(model_name)
        enc_hidden   = self.encoder.hidden_size

        # ── Stage 2: shared projection FNN ──────────────────────────────────
        self.projection = nn.Sequential(
            nn.Linear(enc_hidden, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # ── Stage 3: attention fusion ────────────────────────────────────────
        self.attention_fusion = AttentionFusion(hidden_dim)

        # ── Stage 4: final classifier ────────────────────────────────────────
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_labels),
        )

    def encode_question(self, inputs: dict) -> torch.Tensor:
        """Run encoder + projection for one question slot."""
        cls_repr   = self.encoder(**inputs)          # (batch, enc_hidden)
        projected  = self.projection(cls_repr)       # (batch, hidden_dim)
        return projected

    def forward(self, q1_inputs, q2_inputs, q3_inputs):
        """
        Returns:
          logits          : (batch, num_labels)
          attention_weights: (batch, 3)  — question importance scores
        """
        # Stage 1 + 2: encode each question slot
        r1 = self.encode_question(q1_inputs)   # (batch, hidden_dim)
        r2 = self.encode_question(q2_inputs)
        r3 = self.encode_question(q3_inputs)

        # Stage 3: stack → attention
        stacked = torch.stack([r1, r2, r3], dim=1)          # (batch, 3, hidden_dim)
        fused, attention_weights = self.attention_fusion(stacked)

        # Stage 4: classify
        logits = self.classifier(fused)                       # (batch, num_labels)

        return logits, attention_weights


# ─────────────────────────────────────────────────────────────────────────────
# 6.  MOVE BATCH TO DEVICE HELPER
# ─────────────────────────────────────────────────────────────────────────────

def to_device(inputs_dict: dict, device: str) -> dict:
    return {k: v.to(device) for k, v in inputs_dict.items()}


# ─────────────────────────────────────────────────────────────────────────────
# 7.  TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────

def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, total_correct, total_samples = 0.0, 0, 0

    for batch in tqdm(loader, desc="  Train", leave=False):
        q1_inputs, q2_inputs, q3_inputs, labels = batch

        q1_inputs = to_device(q1_inputs, device)
        q2_inputs = to_device(q2_inputs, device)
        q3_inputs = to_device(q3_inputs, device)
        labels    = labels.to(device)

        # Debug guard: ensure labels are valid integers in [0, num_labels)
        assert labels.dtype == torch.long, "Labels must be torch.long"
        assert labels.min() >= 0 and labels.max() < CFG["num_labels"], \
            f"Label out of range: min={labels.min()}, max={labels.max()}"

        optimizer.zero_grad()

        logits, attn_weights = model(q1_inputs, q2_inputs, q3_inputs)
        loss = criterion(logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()

        preds = logits.argmax(dim=-1)
        total_loss    += loss.item() * labels.size(0)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    return avg_loss, accuracy


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0

    for batch in tqdm(loader, desc="  Eval ", leave=False):
        q1_inputs, q2_inputs, q3_inputs, labels = batch

        q1_inputs = to_device(q1_inputs, device)
        q2_inputs = to_device(q2_inputs, device)
        q3_inputs = to_device(q3_inputs, device)
        labels    = labels.to(device)

        logits, _ = model(q1_inputs, q2_inputs, q3_inputs)
        loss = criterion(logits, labels)

        preds = logits.argmax(dim=-1)
        total_loss    += loss.item() * labels.size(0)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    return total_loss / total_samples, total_correct / total_samples


def train(model, train_loader, val_loader, cfg):
    device    = cfg["device"]
    model     = model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"],
                                  weight_decay=0.01)

    total_steps   = len(train_loader) * cfg["num_epochs"]
    warmup_steps  = int(total_steps * cfg["warmup_ratio"])

    # Linear warmup + linear decay scheduler
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    criterion = nn.CrossEntropyLoss()

    print("\n" + "="*60)
    print("TRAINING")
    print("="*60)

    for epoch in range(1, cfg["num_epochs"] + 1):
        print(f"\nEpoch {epoch}/{cfg['num_epochs']}")

        train_loss, train_acc = train_epoch(
            model, train_loader, optimizer, scheduler, criterion, device)
        val_loss, val_acc     = eval_epoch(model, val_loader, criterion, device)

        print(f"  Train Loss: {train_loss:.4f}  |  Train Acc: {train_acc:.4f}")
        print(f"  Val   Loss: {val_loss:.4f}  |  Val   Acc: {val_acc:.4f}")

    print("\n[TRAINING COMPLETE]")
    return model


# ─────────────────────────────────────────────────────────────────────────────
# 8.  INFERENCE + INTERPRETABILITY
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def predict(model, tokenizer, claim: str,
            q1: str, ev1: str,
            q2: str, ev2: str,
            q3: str, ev3: str,
            cfg: dict):
    """
    Single-sample prediction.
    Returns predicted label string + question attention weights.
    """
    model.eval()
    device = cfg["device"]
    ml     = cfg["max_length"]

    def enc(q, e):
        d = tokenize_pair(tokenizer, claim, q, e, ml)
        return {k: v.unsqueeze(0).to(device) for k, v in d.items()}

    q1_in = enc(q1, ev1)
    q2_in = enc(q2, ev2)
    q3_in = enc(q3, ev3)

    logits, attn_weights = model(q1_in, q2_in, q3_in)

    pred_id    = logits.argmax(dim=-1).item()
    pred_label = ID2LABEL[pred_id]
    probs      = F.softmax(logits, dim=-1).squeeze().cpu().numpy()
    weights    = attn_weights.squeeze().cpu().numpy()

    print("\n── PREDICTION ──────────────────────────────────────")
    print(f"  Claim         : {claim[:80]}")
    print(f"  Predicted     : {pred_label}")
    print(f"  Class probs   : { {ID2LABEL[i]: f'{p:.3f}' for i, p in enumerate(probs)} }")
    print(f"  Q attention   : Q1={weights[0]:.3f}  Q2={weights[1]:.3f}  Q3={weights[2]:.3f}")
    print("────────────────────────────────────────────────────")

    return pred_label, weights


# ─────────────────────────────────────────────────────────────────────────────
# 9.  FULL PIPELINE (entry point)
# ─────────────────────────────────────────────────────────────────────────────

def build_and_train(df: pd.DataFrame, cfg: dict = CFG):
    """
    End-to-end: preprocess → tokenize → dataset → model → train.

    Parameters
    ----------
    df  : raw dataframe with columns: claim, column (or q1/q2/q3), label
    cfg : configuration dict

    Returns
    -------
    model, tokenizer
    """

    # 1. Preprocess
    print("\n[1/5] Preprocessing dataframe ...")
    df_clean = preprocess_dataframe(df)
    assert len(df_clean) > 0, "No valid rows after preprocessing!"

    # 2. Split
    print("[2/5] Splitting train / validation ...")
    train_df, val_df = preprocess_dataframe(df[df.category == 'train']), preprocess_dataframe(df[df.category == 'validation'])
    print(f"  Train: {len(train_df)}  |  Val: {len(val_df)}")

    # 3. Tokenizer + Datasets
    print(f"[3/5] Loading tokenizer: {cfg['model_name']} ...")
    tokenizer = AutoTokenizer.from_pretrained(cfg["model_name"])

    train_ds = FactVerificationDataset(train_df, tokenizer, cfg["max_length"])
    val_ds   = FactVerificationDataset(val_df,   tokenizer, cfg["max_length"])

    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"],
                              shuffle=True,  collate_fn=collate_fn, num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=cfg["batch_size"],
                              shuffle=False, collate_fn=collate_fn, num_workers=0)

    # 4. Model
    print("[4/5] Building model ...")
    model = FactVerificationModel(
        model_name = cfg["model_name"],
        hidden_dim = cfg["hidden_dim"],
        num_labels = cfg["num_labels"],
        dropout    = cfg["dropout"],
    )
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Trainable parameters: {n_params:,}")

    # 5. Train
    print("[5/5] Starting training ...")
    model = train(model, train_loader, val_loader, cfg)

    return model, tokenizer



model, tokenizer = build_and_train(df_train, CFG)

# ── Demo prediction with interpretability ─────────────────────────────
predict(
    model, tokenizer,
    claim = "Obamacare created more jobs than any other policy.",
    q1    = "Did Obamacare affect employment?",
    ev1   = "Studies show Obamacare had mixed effects on job creation.",
    q2    = "What was the job impact of the ACA?",
    ev2   = "The ACA led to some part-time job increases but also losses.",
    q3    = "Has any policy created more jobs than Obamacare?",
    ev3   = "Several economic policies have been credited with larger job gains.",
    cfg   = CFG,
)

[CONFIG] Using device: cuda

[1/5] Preprocessing dataframe ...
[PREPROCESS] Kept 12935 rows, skipped 0 (unknown labels).
[2/5] Splitting train / validation ...
[PREPROCESS] Kept 9872 rows, skipped 0 (unknown labels).
[PREPROCESS] Kept 3063 rows, skipped 0 (unknown labels).
  Train: 9872  |  Val: 3063
[3/5] Loading tokenizer: bert-base-uncased ...
[4/5] Building model ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Trainable parameters: 109,680,643
[5/5] Starting training ...

TRAINING

Epoch 1/3


  Train Loss: 0.8370  |  Train Acc: 0.6188
  Val   Loss: 0.7750  |  Val   Acc: 0.6376

Epoch 2/3


  Train Loss: 0.6737  |  Train Acc: 0.7053
  Val   Loss: 0.7811  |  Val   Acc: 0.6618

Epoch 3/3


  Train Loss: 0.4602  |  Train Acc: 0.8190
  Val   Loss: 0.8775  |  Val   Acc: 0.6605

[TRAINING COMPLETE]

── PREDICTION ──────────────────────────────────────
  Claim         : Obamacare created more jobs than any other policy.
  Predicted     : False
  Class probs   : {'False': '0.661', 'True': '0.019', 'Half True/False': '0.319'}
  Q attention   : Q1=0.427  Q2=0.264  Q3=0.310
────────────────────────────────────────────────────


('False', array([0.42653337, 0.2639589 , 0.3095077 ], dtype=float32))

In [ ]:
import copy

# Define the hyperparameter search space
param_grid = {
    'lr': [1e-5, 2e-5, 5e-5],
    'batch_size': [8, 16, 32]
}

best_accuracy = 0.0
best_cfg = None
best_model = None
best_tokenizer = None

print("\n[HYPERPARAMETER TUNING STARTING]")
for lr in param_grid['lr']:
    for batch_size in param_grid['batch_size']:
        print(f"\n--- Testing LR: {lr}, Batch Size: {batch_size} ---")

        # Create a temporary config for this run
        current_cfg = copy.deepcopy(CFG)
        current_cfg['lr'] = lr
        current_cfg['batch_size'] = batch_size

        # Build and train the model with the current hyperparameters
        # Temporarily redirect stdout to suppress verbose training output during tuning
        import sys
        from io import StringIO

        old_stdout = sys.stdout
        redirected_output = StringIO()
        sys.stdout = redirected_output

        try:
            model, tokenizer = build_and_train(df_train, current_cfg)
        finally:
            sys.stdout = old_stdout # Restore stdout
            print(redirected_output.getvalue())

        # Evaluate the model on the validation set to get accuracy
        # Note: eval_epoch is defined in the previous cell
        train_loader = DataLoader(FactVerificationDataset(preprocess_dataframe(df[df.category == 'train']), tokenizer, current_cfg['max_length']), batch_size=current_cfg['batch_size'], shuffle=False, collate_fn=collate_fn, num_workers=0)
        val_loader   = DataLoader(FactVerificationDataset(preprocess_dataframe(df[df.category == 'validation']), tokenizer, current_cfg['max_length']), batch_size=current_cfg['batch_size'], shuffle=False, collate_fn=collate_fn, num_workers=0)

        val_loss, val_acc = eval_epoch(model, val_loader, nn.CrossEntropyLoss(), current_cfg['device'])
        print(f"  Validation Accuracy for LR={lr}, Batch Size={batch_size}: {val_acc:.4f}")

        # Check if this is the best model so far
        if val_acc > best_accuracy:
            best_accuracy = val_acc
            best_cfg = current_cfg
            best_model = model
            best_tokenizer = tokenizer

print("\n[HYPERPARAMETER TUNING COMPLETE]")
print(f"Best Validation Accuracy: {best_accuracy:.4f}")
print(f"Best Configuration: {best_cfg}")

# You can now use best_model and best_tokenizer for further evaluation or inference
# For example, to predict with the best model:
# predict(
#     best_model, best_tokenizer,
#     claim = "Obamacare created more jobs than any other policy.",
#     q1    = "Did Obamacare affect employment?",
#     ev1   = "Studies show Obamacare had mixed effects on job creation.",
#     q2    = "What was the job impact of the ACA?",
#     ev2   = "The ACA led to some part-time job increases but also losses.",
#     q3    = "Has any policy created more jobs than Obamacare?",
#     ev3   = "Several economic policies have been credited with larger job gains.",
#     cfg   = best_cfg,
# )

In [ ]:
clean_test_df = preprocess_dataframe(df_test)

[PREPROCESS] Kept 2480 rows, skipped 0 (unknown labels).


In [ ]:
clean_test_df

,claim,q1,ev1,q2,ev2,q3,ev3,label_id
0,meat and dairy products will be banned by 2030...,Which products will be banned from the EU in t...,This suggests that specialisation in importing...,Which products will be banned from the meat an...,"Dec 25, 2022 ... No, i doubt that banning meat...",Which product will be banned by the year 2030?,"Goal: Reduce illness, disability, and death re...",0
1,""" Over the past five years..more than R1.3 tri...",Which country has invested more than R1.3 tril...,"Oct 15, 2020 ... • unlock more than R1 trillio...",Which country has invested at least R1.3 trill...,"Feb 25, 2024 ... least three more countries ha...",Which country has invested R1.3 trillion or mo...,Public infrastructure spending over the next t...,1
2,""" Singapore... got its independence five years...","Which country's GDP per capita was $1,951?",According to the World Bank’s most recent data...,Who has a real GDP per capita of over half a b...,"Using the NHE deflator, real national health e...","Which country had $1,951 in the year 2017?",It accounts for all sales across the country a...,1
3,"""#FISAMemo shows real collusion between Dem op...",Who used #FISAMemo to spy on Trump's campaign?,#FISAMemo · #FreedomCaucus · #MichealCohn · #S...,Who spied on the #Trump campaign & who interfe...,McGahn II White House counsel Stephen Miller T...,Who showed collusion between Dem operatives & ...,FBI and DOJ officials did not omit material in...,0
4,"""(Flint, Mich., is) paying three times more fo...",Which city is paying three times more for pois...,"Until around the 1950s, lead pipes were common...",Who is paying three times more for poison wate...,"Maddow continued on, laying ultimate blame on ...",Who is paying three times more for poison wate...,[applause] What is absolute incredible to me i...,1
...,...,...,...,...,...,...,...,...
2475,“coronavirus itself is responsible for just 6%...,Which disease is responsible for 6% or more of...,The number of deaths that mention one or more ...,Which virus is responsible for more than half ...,The number of deaths that mention one or more ...,Which virus is responsible for 6% or more of a...,The number of deaths that mention one or more ...,0
2476,“the CGG CGG coding [...] does not appear in n...,Which gene has a furin cleavage site?,Alliance https://doi.org/10.26508/lsa.20200078...,Which gene had a CGG CGG coding site that was ...,"Oct 6, 2016 ... ... not show CGG expansion but...",Which virus was not transmitted from animals t...,Animals and COVID-19 - The risk of animals spr...,0
2477,“… in contrast to the lifetime risk in develop...,Which country has 1 in 4900 life expectancy?,"May 18, 2020 ... ... is 1 in 22, in contrast t...",Which country has 1 in 4900 lifetime risk?,"May 18, 2020 ... ... is 1 in 22, in contrast t...",Which country has a risk of 1 in 4900 years?,"Between 2005 and 2015, it is estimated that ov...",0
2478,”The post-1994 governments have cumulatively b...,Which countries have bought less than 7% of th...,Verdict: Incorrect The claim that the governme...,Which government has purchased less than 7% of...,The state has bought 8.2 million hectares – or...,Who has purchased less than 7% of the targeted...,The state has bought 8.2 million hectares – or...,0


In [ ]:
clean_test_df['pred'] = clean_test_df.apply(
    lambda row: predict(
        model=model,
        tokenizer=tokenizer,
        claim=row['claim'],
        q1=row['q1'], ev1=row['ev1'],
        q2=row['q2'], ev2=row['ev2'],
        q3=row['q3'], ev3=row['ev3'],
        cfg=CFG
    )[0], # [0] to get only the predicted label, not the weights
    axis=1
)

Streaming output truncated to the last 5000 lines.
  Q attention   : Q1=0.317  Q2=0.373  Q3=0.310
────────────────────────────────────────────────────

── PREDICTION ──────────────────────────────────────
  Claim         : Says that under a new program jail "time for non-violent, mentally ill offenders
  Predicted     : Half True/False
  Class probs   : {'False': '0.060', 'True': '0.168', 'Half True/False': '0.772'}
  Q attention   : Q1=0.345  Q2=0.294  Q3=0.361
────────────────────────────────────────────────────

── PREDICTION ──────────────────────────────────────
  Claim         : Says the 1956 Republican Party platform supported equal pay, the minimum wage, a
  Predicted     : True
  Class probs   : {'False': '0.033', 'True': '0.882', 'Half True/False': '0.085'}
  Q attention   : Q1=0.343  Q2=0.321  Q3=0.336
────────────────────────────────────────────────────

── PREDICTION ──────────────────────────────────────
  Claim         : Says the 1995 Chicago heat wave was “the largest s

In [ ]:
clean_test_df

,claim,q1,ev1,q2,ev2,q3,ev3,label_id,pred
0,meat and dairy products will be banned by 2030...,Which products will be banned from the EU in t...,This suggests that specialisation in importing...,Which products will be banned from the meat an...,"Dec 25, 2022 ... No, i doubt that banning meat...",Which product will be banned by the year 2030?,"Goal: Reduce illness, disability, and death re...",0,False
1,""" Over the past five years..more than R1.3 tri...",Which country has invested more than R1.3 tril...,"Oct 15, 2020 ... • unlock more than R1 trillio...",Which country has invested at least R1.3 trill...,"Feb 25, 2024 ... least three more countries ha...",Which country has invested R1.3 trillion or mo...,Public infrastructure spending over the next t...,1,True
2,""" Singapore... got its independence five years...","Which country's GDP per capita was $1,951?",According to the World Bank’s most recent data...,Who has a real GDP per capita of over half a b...,"Using the NHE deflator, real national health e...","Which country had $1,951 in the year 2017?",It accounts for all sales across the country a...,1,True
3,"""#FISAMemo shows real collusion between Dem op...",Who used #FISAMemo to spy on Trump's campaign?,#FISAMemo · #FreedomCaucus · #MichealCohn · #S...,Who spied on the #Trump campaign & who interfe...,McGahn II White House counsel Stephen Miller T...,Who showed collusion between Dem operatives & ...,FBI and DOJ officials did not omit material in...,0,False
4,"""(Flint, Mich., is) paying three times more fo...",Which city is paying three times more for pois...,"Until around the 1950s, lead pipes were common...",Who is paying three times more for poison wate...,"Maddow continued on, laying ultimate blame on ...",Who is paying three times more for poison wate...,[applause] What is absolute incredible to me i...,1,Half True/False
...,...,...,...,...,...,...,...,...,...
2475,“coronavirus itself is responsible for just 6%...,Which disease is responsible for 6% or more of...,The number of deaths that mention one or more ...,Which virus is responsible for more than half ...,The number of deaths that mention one or more ...,Which virus is responsible for 6% or more of a...,The number of deaths that mention one or more ...,0,False
2476,“the CGG CGG coding [...] does not appear in n...,Which gene has a furin cleavage site?,Alliance https://doi.org/10.26508/lsa.20200078...,Which gene had a CGG CGG coding site that was ...,"Oct 6, 2016 ... ... not show CGG expansion but...",Which virus was not transmitted from animals t...,Animals and COVID-19 - The risk of animals spr...,0,False
2477,“… in contrast to the lifetime risk in develop...,Which country has 1 in 4900 life expectancy?,"May 18, 2020 ... ... is 1 in 22, in contrast t...",Which country has 1 in 4900 lifetime risk?,"May 18, 2020 ... ... is 1 in 22, in contrast t...",Which country has a risk of 1 in 4900 years?,"Between 2005 and 2015, it is estimated that ov...",0,Half True/False
2478,”The post-1994 governments have cumulatively b...,Which countries have bought less than 7% of th...,Verdict: Incorrect The claim that the governme...,Which government has purchased less than 7% of...,The state has bought 8.2 million hectares – or...,Who has purchased less than 7% of the targeted...,The state has bought 8.2 million hectares – or...,0,True


In [ ]:
clean_test_df['pred_lable'] = clean_test_df['pred'].map(LABEL2ID)

In [ ]:
clean_test_df

,claim,q1,ev1,q2,ev2,q3,ev3,label_id,pred,pred_lable
0,meat and dairy products will be banned by 2030...,Which products will be banned from the EU in t...,This suggests that specialisation in importing...,Which products will be banned from the meat an...,"Dec 25, 2022 ... No, i doubt that banning meat...",Which product will be banned by the year 2030?,"Goal: Reduce illness, disability, and death re...",0,False,0
1,""" Over the past five years..more than R1.3 tri...",Which country has invested more than R1.3 tril...,"Oct 15, 2020 ... • unlock more than R1 trillio...",Which country has invested at least R1.3 trill...,"Feb 25, 2024 ... least three more countries ha...",Which country has invested R1.3 trillion or mo...,Public infrastructure spending over the next t...,1,True,1
2,""" Singapore... got its independence five years...","Which country's GDP per capita was $1,951?",According to the World Bank’s most recent data...,Who has a real GDP per capita of over half a b...,"Using the NHE deflator, real national health e...","Which country had $1,951 in the year 2017?",It accounts for all sales across the country a...,1,True,1
3,"""#FISAMemo shows real collusion between Dem op...",Who used #FISAMemo to spy on Trump's campaign?,#FISAMemo · #FreedomCaucus · #MichealCohn · #S...,Who spied on the #Trump campaign & who interfe...,McGahn II White House counsel Stephen Miller T...,Who showed collusion between Dem operatives & ...,FBI and DOJ officials did not omit material in...,0,False,0
4,"""(Flint, Mich., is) paying three times more fo...",Which city is paying three times more for pois...,"Until around the 1950s, lead pipes were common...",Who is paying three times more for poison wate...,"Maddow continued on, laying ultimate blame on ...",Who is paying three times more for poison wate...,[applause] What is absolute incredible to me i...,1,Half True/False,2
...,...,...,...,...,...,...,...,...,...,...
2475,“coronavirus itself is responsible for just 6%...,Which disease is responsible for 6% or more of...,The number of deaths that mention one or more ...,Which virus is responsible for more than half ...,The number of deaths that mention one or more ...,Which virus is responsible for 6% or more of a...,The number of deaths that mention one or more ...,0,False,0
2476,“the CGG CGG coding [...] does not appear in n...,Which gene has a furin cleavage site?,Alliance https://doi.org/10.26508/lsa.20200078...,Which gene had a CGG CGG coding site that was ...,"Oct 6, 2016 ... ... not show CGG expansion but...",Which virus was not transmitted from animals t...,Animals and COVID-19 - The risk of animals spr...,0,False,0
2477,“… in contrast to the lifetime risk in develop...,Which country has 1 in 4900 life expectancy?,"May 18, 2020 ... ... is 1 in 22, in contrast t...",Which country has 1 in 4900 lifetime risk?,"May 18, 2020 ... ... is 1 in 22, in contrast t...",Which country has a risk of 1 in 4900 years?,"Between 2005 and 2015, it is estimated that ov...",0,Half True/False,2
2478,”The post-1994 governments have cumulatively b...,Which countries have bought less than 7% of th...,Verdict: Incorrect The claim that the governme...,Which government has purchased less than 7% of...,The state has bought 8.2 million hectares – or...,Who has purchased less than 7% of the targeted...,The state has bought 8.2 million hectares – or...,0,True,1


In [ ]:
from sklearn.metrics import classification_report

y_true = clean_test_df['label_id']
y_pred = clean_test_df['pred_lable']

# Generate and print the classification report
report = classification_report(y_true, y_pred, target_names=[ID2LABEL[i] for i in sorted(ID2LABEL.keys())])
print(report)

                 precision    recall  f1-score   support

          False       0.80      0.77      0.79      1413
           True       0.44      0.51      0.47       472
Half True/False       0.45      0.45      0.45       595

       accuracy                           0.64      2480
      macro avg       0.57      0.58      0.57      2480
   weighted avg       0.65      0.64      0.65      2480



In [ ]:
clean_test_df.to_json("/content/drive/MyDrive/claim_verification_final/final_result_reranked_temporal+page_content_refined.json", orient='records', indent=2)